In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
# Leemos los csv creados en mainpisa y guardados en la carpeta datos
df_valencia25=pd.read_csv('data/pisa_valencia25.csv')
df_valencia22=pd.read_csv('data/pisa_valencia22.csv')
df_valencia18=pd.read_csv('data/pisa_valencia18.csv')

In [12]:
# =============================================================
# 1. PROCESAMIENTO DE 2022 (Horas en pantalla en el centro)
# =============================================================
colestudio="ST326Q05JA"
datos22 = df_valencia22.dropna(subset=[colestudio, "W_FSTUWT"]).copy()

# Diccionario de traducción de códigos a horas aproximadas (punto medio)
mapeo_22 = {1: 0, 2: 0.5, 3: 1.5, 4: 2.5, 5: 3.5, 6: 4.5, 7: 5.5, 8: 6.5, 9: 7.5}
datos22["horas"] = datos22[colestudio].map(mapeo_22)

# Producto de horas por el peso muestral para 2022
datos22["horas_por_peso"] = datos22["horas"] * datos22["W_FSTUWT"]

# Media ponderada de horas en el centro (2022)
media_ponderada_22 = datos22["horas_por_peso"].sum() / datos22["W_FSTUWT"].sum()


# =============================================================
# 2. PROCESAMIENTO DE 2025 (Horas en pantalla en el centro)
# =============================================================
# *Nota: Reemplaza 'NOMBRE_COLUMNA_2025_CENTRO' por la variable real de 2025 equivalente a ST326Q01JA*
datos25 = df_valencia25.dropna(subset=[colestudio, "W_FSTUWT"]).copy()

# Si en 2025 los códigos numéricos o textos son distintos, ajusta este mapeo o lógica:
# (Ejemplo asumiendo un mapeo numérico similar al de 2022 o ajustado al codebook de 2025)
mapeo_25 = {1: 0, 2: 0.5, 3: 1.5, 4: 2.5, 5: 3.5, 6: 4.5, 7: 5.5, 8: 6.5, 9: 7.5}
datos25["horas"] = datos25[colestudio].map(mapeo_25)

# Producto de horas por el peso muestral para 2025
datos25["horas_por_peso"] = datos25["horas"] * datos25["W_FSTUWT"]

# Media ponderada de horas en el centro (2025)
media_ponderada_25 = datos25["horas_por_peso"].sum() / datos25["W_FSTUWT"].sum()


# =============================================================
# 3. TABLA COMPARATIVA FINAL ENTRE 2022 Y 2025
# =============================================================
comparacion_horas_centro = pd.DataFrame({
    "Año": ["2022", "2025"],
    "Media Horas en Centro (Ponderada)": [media_ponderada_22, media_ponderada_25]
}).set_index("Año")

print(comparacion_horas_centro.round(2))

KeyError: ['ST326Q05JA']

In [ ]:
# 1) Quitamos los alumnos sin respuesta en la pregunta de horas
datos = df_valencia25.dropna(subset=["ST326Q01JA"]).copy()

# 2) Tu diccionario: cada código de respuesta -> horas aproximadas (punto medio del tramo)
mapeo = {1: 0, 2: 0.5, 3: 1.5, 4: 2.5, 5: 3.5, 6: 4.5, 7: 5.5, 8: 6.5, 9: 7.5}

# 3) Creamos la columna "horas" traduciendo cada código con el diccionario
datos["horas"] = datos["ST326Q01JA"].map(mapeo)
datos["pv_por_peso"] = datos["PV1READ"] * datos["W_FSTUWT"]

# # 4) Comprobaciones
# print(len(datos))                          # alumnos que quedan (debería ser 1473)
# print(datos["horas"].isna().sum())         # debería ser 0 (todos los códigos se tradujeron)
# print(datos["horas"].value_counts().sort_index()) 

# Agrupamos a los alumnos por su valor de horas y calculamos,
# para cada grupo, cuántos son y cuál es su lectura media (con PV1READ)
resumen = datos.groupby("horas")["PV1READ"].agg(["count", "mean"])

# Redondeamos para leerlo mejor
print(resumen.round(1))

# 2) Para cada grupo de horas, sumamos (nota x peso) y sumamos los pesos
sumas = datos.groupby("horas")[["pv_por_peso", "W_FSTUWT"]].sum()

# 3) La media ponderada es: suma(nota x peso) / suma(pesos)
sumas["media_ponderada"] = sumas["pv_por_peso"] / sumas["W_FSTUWT"]

# 4) Comparamos la media sin pesos (paso 2) con la media con pesos
comparacion = pd.DataFrame({
    "sin_pesos": resumen["mean"],
    "con_pesos": sumas["media_ponderada"],
})
print(comparacion.round(1))

       count   mean
horas              
0.0      314  442.0
0.5      558  437.3
1.5      282  432.2
2.5      112  422.8
3.5       64  451.7
4.5       38  452.3
5.5       36  486.9
6.5       35  462.5
7.5       34  436.6
       sin_pesos  con_pesos
horas                      
0.0        442.0      437.8
0.5        437.3      432.1
1.5        432.2      431.9
2.5        422.8      422.1
3.5        451.7      451.4
4.5        452.3      443.5
5.5        486.9      487.5
6.5        462.5      459.0
7.5        436.6      434.1


In [7]:
# 1) Quitamos los alumnos sin respuesta en la pregunta de horas
datos22 = df_valencia22.dropna(subset=["ST326Q01JA"]).copy()

# 2) Tu diccionario: cada código de respuesta -> horas aproximadas (punto medio del tramo)
mapeo = {1: 0, 2: 0.5, 3: 1.5, 4: 2.5, 5: 3.5, 6: 4.5, 7: 5.5, 8: 6.5, 9: 7.5}

# 3) Creamos la columna "horas" traduciendo cada código con el diccionario
datos22["horas"] = datos22["ST326Q01JA"].map(mapeo)
datos22["pv_por_peso"] = datos22["PV1READ"] * datos22["W_FSTUWT"]

# # 4) Comprobaciones
# print(len(datos))                          # alumnos que quedan (debería ser 1473)
# print(datos["horas"].isna().sum())         # debería ser 0 (todos los códigos se tradujeron)
# print(datos["horas"].value_counts().sort_index()) 

# Agrupamos a los alumnos por su valor de horas y calculamos,
# para cada grupo, cuántos son y cuál es su lectura media (con PV1READ)
resumen22 = datos22.groupby("horas")["PV1READ"].agg(["count", "mean"])

# Redondeamos para leerlo mejor
print(resumen22.round(1))

# 2) Para cada grupo de horas, sumamos (nota x peso) y sumamos los pesos
sumas22 = datos22.groupby("horas")[["pv_por_peso", "W_FSTUWT"]].sum()

# 3) La media ponderada es: suma(nota x peso) / suma(pesos)
sumas22["media_ponderada"] = sumas22["pv_por_peso"] / sumas22["W_FSTUWT"]

# 4) Comparamos la media sin pesos (paso 2) con la media con pesos
comparacion22 = pd.DataFrame({
    "con_pesos22": sumas22["media_ponderada"],
    "con_pesos25": sumas["media_ponderada"],
})
print(comparacion22.round(1))

       count   mean
horas              
0.0      325  479.0
0.5      627  490.7
1.5      288  490.4
2.5      140  510.8
3.5       50  503.4
4.5       44  512.0
5.5       41  514.2
6.5       32  485.3
7.5       32  483.7
       con_pesos22  con_pesos25
horas                          
0.0          472.8        437.8
0.5          486.1        432.1
1.5          485.6        431.9
2.5          506.0        422.1
3.5          496.8        451.4
4.5          509.0        443.5
5.5          506.9        487.5
6.5          483.8        459.0
7.5          478.0        434.1


In [3]:
# --------------------- 1. CÁLCULO CORREGIDO PARA 2018 -------------------------------
mapeo_st176 = {
    "I don't know what it is": 0,
    "Never or almost never": 0,
    "Several times a month": 0,
    "Several times a week": 0,
    "Several times a day": 1,
}

cols_176 = [
    c for c in df_valencia18.columns if c.startswith("ST176") and c.endswith("IA")
]
df_mapeado = df_valencia18[cols_176].replace(mapeo_st176)
df_valencia18["digital_18_bin"] = (df_mapeado.max(axis=1) >= 1).astype(float)

# Limpieza estricta de nulos
d18 = df_valencia18.dropna(subset=["W_FSTUWT", "digital_18_bin"]).copy()

# Cálculo correcto del porcentaje ponderado: (Suma de (binaria * peso)) / (Suma de pesos) * 100
pct_18_binario = (
    100 * (d18["digital_18_bin"] * d18["W_FSTUWT"]).sum() / d18["W_FSTUWT"].sum()
)


# -------------------------------------------------------------
# 2. CÁLCULO CORREGIDO PARA 2025
# -------------------------------------------------------------
mapeo_horas_2025 = {
    "None": 0,
    "Up to 1 hour": 0,
    "More than 1 hour and up to 2 hours": 0,
    "More than 2 hours and up to 3 hours": 1,
    "More than 3 hours and up to 4 hours": 1,
    "More than 4 hours and up to 5 hours": 1,
    "More than 5 hours and up to 6 hours": 1,
    "More than 6 hours and up to 7 hours": 1,
    "More than 7 hours": 1,
}


df_valencia25["digital_25_bin"] = df_valencia25["ST326Q01JA"].replace(
    mapeo_horas_2025
)

# Limpieza estricta de nulos
d25 = df_valencia25.dropna(subset=["W_FSTUWT", "digital_25_bin"])

# Cálculo correcto del porcentaje ponderado para 2025
pct_25_binario = (
    100 * (d25["digital_25_bin"] * d25["W_FSTUWT"]).sum() / d25["W_FSTUWT"].sum()
)

print(
    f"Porcentaje con alta intensidad digital en 2018: {round(pct_18_binario, 1)}%"
)
print(
    f"Porcentaje con alta intensidad digital en 2025: {round(pct_25_binario, 1)}%"
)

Porcentaje con alta intensidad digital en 2018: 94.1%
Porcentaje con alta intensidad digital en 2025: 280.3%


C:\Users\TB Val 03\AppData\Local\Temp\ipykernel_23976\719389530.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_valencia18["digital_18_bin"] = (df_mapeado.max(axis=1) >= 1).astype(float)
